# Zero-shot Thai Retrieval Demo

Notebook นี้สรุป Phase 1-3 สำหรับโปรเจกต์ Thai Vision-Language Retrieval โดยเน้น 3 เรื่องหลัก

1. เปลี่ยน text stack ไปใช้ multilingual pre-trained encoder
2. เพิ่ม learned 2D positional encoding ให้ vision tokens ก่อนเข้า Mamba
3. เทรนด้วย contrastive retrieval objective และวัดผลด้วย Recall@K

## 1. ติดตั้งสภาพแวดล้อม Colab/Linux และตรวจสอบ `mamba-ssm`

- แนะนำให้รันบน Colab หรือ Linux ที่มี NVIDIA GPU
- บน macOS โค้ดจะ fallback ไปใช้ selective SSM stub แทน `mamba-ssm`
- Runtime ที่เหมาะควรมี CUDA พร้อมใช้งานก่อนเริ่มเทรนหรือ evaluate

In [ ]:
# Section 1: Colab/Linux setup and real Mamba verification
# Run this cell on Linux/Colab with an NVIDIA GPU.

# %pip install -r requirements.txt

import importlib.util
import platform
import torch

print({'platform': platform.platform(), 'cuda_available': torch.cuda.is_available(), 'cuda_device_count': torch.cuda.device_count()})
print({'mamba_ssm_installed': importlib.util.find_spec('mamba_ssm') is not None})

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('CUDA is not available. Use Colab/Linux GPU for the real mamba-ssm path.')

## 2. โหลด Multilingual Tokenizer และ Text Encoder สำหรับภาษาไทย
## 3. ทดสอบการตัดคำไทยและตรวจสอบ Text Embeddings

ด้านล่างจะโหลด `distilbert-base-multilingual-cased` ผ่านโค้ดในโปรเจกต์ แล้วตรวจทั้ง tokenization และ output embedding shapes เพื่อยืนยันว่าภาษาไทยไม่ได้ถูกย่อยเป็นวรรณยุกต์แบบ tokenizer เดิมอีกต่อไป

In [ ]:
# Sections 2-3: tokenizer and multilingual text encoder sanity check

from pathlib import Path

from models import build_svlb_from_config
from utils import MultilingualTokenizer, load_config

config = load_config('configs/default.yaml')
tokenizer = MultilingualTokenizer(
    model_name=config['model']['text']['pretrained_model_name'],
    max_length=config['model']['text']['max_length'],
)
model = build_svlb_from_config(config)
text_encoder = model.text_backbone

thai_examples = [
    'คนขี่จักรยานหน้าวัด',
    'เด็กชายถือร่มในขณะที่ยืนถัดจากปศุสัตว์',
    'รถบรรทุกจอดอยู่ริมถนน',
]

encoded = tokenizer.batch_encode(thai_examples)
print({'input_ids_shape': tuple(encoded['input_ids'].shape), 'attention_mask_shape': tuple(encoded['attention_mask'].shape)})
for text, token_ids in zip(thai_examples, encoded['input_ids']):
    tokens = tokenizer.tokenizer.convert_ids_to_tokens(token_ids.tolist())
    print({'text': text, 'tokens': tokens[:16]})

with torch.inference_mode():
    text_embedding, text_tokens = model.encode_text_embedding(encoded['input_ids'], encoded['attention_mask'])

print({'text_tokens_shape': tuple(text_tokens.shape), 'text_embedding_shape': tuple(text_embedding.shape)})

## 4. เพิ่ม 2D Positional Encoding ให้ Vision Tokens
## 5. ประกอบ Vision-Language Encoder แบบ Mamba + Transformer
## 6. แก้ `dataset.py` เพื่อใช้ Caption ทั้ง 5 ประโยคของ Thai COCO
## 7. สร้าง Image-Text Batches สำหรับ Contrastive Training
## 8. เปลี่ยน Loss เป็น InfoNCE แบบ CLIP
## 9. เพิ่ม Hard Negative Sampling

เซลล์ถัดไปจะดึงของจริงจาก repo เพื่อเช็ก shape ของ vision tokens, ตรวจว่า learned positional embedding มีขนาด `[1, 49, dim]`, สาธิต collate function และคำนวณ contrastive similarity matrix พร้อม hard-negative term

In [ ]:
# Sections 4-9: vision position encoding, full-caption dataset, collate, InfoNCE, and hard negatives

import torch
from torch.utils.data import DataLoader

from data import ThaiCOCODataset
from train import build_collate_fn, compute_contrastive_loss
from utils import build_image_transform

image_transform = build_image_transform(config['data']['image_size'])
dataset = ThaiCOCODataset(
    split='train',
    transform=image_transform,
    dataset_name=config['train']['dataset_name'],
    use_all_captions=True,
    shuffle_buffer_size=64,
)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    num_workers=0,
    collate_fn=build_collate_fn(tokenizer, config['model']['text']['max_length']),
)

batch = next(iter(dataloader))
output = model(
    images=batch['images'],
    token_ids=batch['input_ids'],
    attention_mask=batch['attention_mask'],
)

print({'vision_pos_shape': tuple(model.vision_position_embedding.shape)})
print({'vision_tokens_shape': tuple(output.vision_tokens.shape), 'text_tokens_shape': tuple(output.text_tokens.shape)})
print({'similarity_shape': tuple(output.similarity_matrix.shape), 'image_embedding_shape': tuple(output.image_embedding.shape)})
print({'unique_image_ids_in_batch': sorted(set(batch['image_ids'].tolist()))})
print({'captions': batch['captions']})

loss, similarity, positive_mask = compute_contrastive_loss(
    model=model,
    images=batch['images'],
    input_ids=batch['input_ids'],
    attention_mask=batch['attention_mask'],
    image_ids=batch['image_ids'],
    device=torch.device('cpu'),
    hard_negative_margin=config['train']['hard_negative_margin'],
    hard_negative_weight=config['train']['hard_negative_weight'],
)
print({'contrastive_loss': float(loss.detach()), 'positive_mask_shape': tuple(positive_mask.shape)})

## 10. เขียน Training Loop บน NVIDIA GPU
## 11. วัดผล Retrieval ด้วย Recall@1, Recall@5, Recall@10
## 12. สร้าง Zero-shot Thai Retrieval Demo
## 13. สร้างข้อความสรุปผลลัพธ์สำหรับใส่ Resume

เซลล์สุดท้ายจะเรียก training/evaluation script ของโปรเจกต์โดยตรง และให้ตัวอย่าง inference กับข้อความไทยหลายประโยค รวมถึงสร้างข้อความสรุปสำหรับใส่ resume

In [ ]:
# Sections 10-13: training commands, Recall@K evaluation, zero-shot retrieval, and resume summary

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

checkpoint_path = Path('checkpoints/svlb_epoch_5.pth')
image_path = Path('samples/example.jpg')
candidate_texts = [
    'คนขี่จักรยานหน้าวัด',
    'รถบรรทุกจอดอยู่ริมถนน',
    'สุนัขนอนอยู่บนโซฟา',
    'เด็กกำลังเล่นฟุตบอลกลางสนาม',
]

print('Train command:')
print('python train.py --config configs/default.yaml --device cuda --batch_size 32 --epochs 5')
print('\nEvaluate command:')
print('python evaluate.py --config configs/default.yaml --checkpoint checkpoints/svlb_epoch_5.pth --split validation --num-images 1000 --device cuda')

if checkpoint_path.is_file() and image_path.is_file():
    from inference import load_config as _unused  # keeps imports local in notebook context
    model = build_svlb_from_config(config)
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    image = Image.open(image_path).convert('RGB')
    image_tensor = image_transform(image).unsqueeze(0)
    encoded = tokenizer.batch_encode(candidate_texts)

    with torch.inference_mode():
        output = model(
            images=image_tensor.expand(len(candidate_texts), -1, -1, -1),
            token_ids=encoded['input_ids'],
            attention_mask=encoded['attention_mask'],
        )
    ranking = pd.DataFrame({'text': candidate_texts, 'score': output.match_logit.tolist()}).sort_values('score', ascending=False)
    display(image)
    display(ranking)
else:
    print('Set checkpoint_path and image_path to existing files before running the zero-shot demo cell.')

resume_summary = [
    'VLM Hybrid (CNN + Mamba + Multilingual Transformer) for Thai Language Understanding',
    'Optimized Vision-Language alignment using Contrastive Learning on Thai-MSCOCO dataset',
    'Evaluated Thai retrieval with Recall@1, Recall@5, and Recall@10 plus zero-shot ranking demos',
]
for line in resume_summary:
    print('-', line)